# 2. Guardrails, built from nothing

Notebook 1 covered finding the right facts. This one is about the two checks either side of the agent: what stops a message reaching it, and what stops its reply reaching the prospect.

No API key needed. I fake the model with a plain function so I can see the control flow without spending anything. Real versions are at the end.


## Why guard the input at all

Start with the naive version. Agent gets the message, agent replies.


In [2]:
def fake_agent(message, entries):
    """Stands in for the LLM. Deliberately a bit too eager, like the real thing."""
    low = message.lower()
    if 'cost' in low or 'price' in low:
        return "It's about $200 a month, but I can check for your setup."
    if entries:
        return entries[0]['answer']
    return "Happy to help -- what would you like to know?"

print(fake_agent('how much does it cost', []))


It's about $200 a month, but I can check for your setup.


There it is. The model produced a price. I never told it one — it came from whatever it absorbed about SaaS pricing in training.

A prospect who receives that will hold me to it. This is the single most expensive thing this system can do, so it shouldn't depend on the model behaving.

## Layer 1: keywords, no model


In [3]:
import re

RESTRICTED = {
    'pricing': ['price', 'pricing', 'cost', 'how much', 'what do you charge', 'quote'],
    'contract_terms': ['contract', 'cancel', 'refund', 'commitment'],
    'legal_compliance': ['gdpr', 'privacy policy', 'record calls', 'liability'],
}
HANDOFF = ['talk to someone', 'real person', 'call me', 'human']

def phrase_in(p, t):
    return re.search(rf'\b{re.escape(p)}\b', t) is not None

def check_keywords(message):
    low = message.lower()
    if any(phrase_in(t, low) for t in HANDOFF):
        return {'allowed': False, 'reason': 'asked for a person', 'layer': 'keywords'}
    for topic, triggers in RESTRICTED.items():
        if any(phrase_in(t, low) for t in triggers):
            return {'allowed': False, 'reason': topic, 'layer': 'keywords'}
    return {'allowed': True, 'layer': 'keywords'}

for m in ['how much does it cost', 'can I talk to someone', 'is this a robot']:
    print(f'{m!r:28} -> {check_keywords(m)}')


'how much does it cost'      -> {'allowed': False, 'reason': 'pricing', 'layer': 'keywords'}
'can I talk to someone'      -> {'allowed': False, 'reason': 'asked for a person', 'layer': 'keywords'}
'is this a robot'            -> {'allowed': True, 'layer': 'keywords'}


Free, instant, and certain. 'How much does it cost' will never reach the agent, on any run, regardless of temperature.

That certainty is the point. If I let a model decide whether a pricing question is close enough to answer, then whether a prospect gets quoted a made-up number depends on a sampled token.

## Where layer 1 fails


In [5]:
for m in ["what's the damage", 'is this gonna break the bank',
          'do I get stuck in something', "what's the tab"]:
    print(f'{m!r:32} -> {check_keywords(m)}')

"what's the damage"              -> {'allowed': True, 'layer': 'keywords'}
'is this gonna break the bank'   -> {'allowed': True, 'layer': 'keywords'}
'do I get stuck in something'    -> {'allowed': True, 'layer': 'keywords'}
"what's the tab"                 -> {'allowed': True, 'layer': 'keywords'}


All four mean pricing or contract terms. None contain a trigger.

I can't enumerate English. So layer 2 is a cheap classifier that catches phrasing -- but it runs *after* the keywords, and it can only ever add refusals.

That ordering is deliberate. If the classifier ran first, or could overturn a keyword block, then 'how much does it cost' would become
probabilistic. It currently isn't, and I want to keep it that way.


In [6]:
def fake_classifier(message):
    """Pretend LLM. Real one is a cheap model with structured output."""
    low = message.lower()
    money_ish = ['damage', 'break the bank', 'tab', 'worth it', 'ballpark']
    lock_in   = ['stuck in', 'tied in', 'locked in']
    if any(w in low for w in money_ish):  return {'category': 'restricted', 'topic': 'pricing'}
    if any(w in low for w in lock_in):    return {'category': 'restricted', 'topic': 'contract_terms'}
    if 'weather' in low or 'game' in low: return {'category': 'off_topic', 'topic': ''}
    return {'category': 'answerable', 'topic': ''}

def check_scope(message):
    kw = check_keywords(message)
    if not kw['allowed']:
        return kw                      # keyword block stands, no model involved
    try:
        result = fake_classifier(message)
    except Exception as e:
        # Fails closed. If the classifier is down, escalate rather than guess.
        return {'allowed': False, 'reason': f'classifier unavailable ({e})', 'layer': 'classifier'}
    if result['category'] == 'restricted':
        return {'allowed': False, 'reason': result['topic'], 'layer': 'classifier'}
    # off_topic is NOT blocked - the agent handles it with a one-line redirect,
    # which is a better reply than silently escalating small talk to a human.
    return {'allowed': True, 'layer': kw['layer'] if result['category'] == 'answerable' else 'classifier'}

for m in ['how much does it cost', "what's the damage", 'do I get stuck in something',
          'is this a robot', "what's the weather"]:
    v = check_scope(m)
    print(f"{m!r:32} allowed={str(v['allowed']):5} layer={v['layer']:11} {v.get('reason','')}")


'how much does it cost'          allowed=False layer=keywords    pricing
"what's the damage"              allowed=False layer=classifier  pricing
'do I get stuck in something'    allowed=False layer=classifier  contract_terms
'is this a robot'                allowed=True  layer=keywords    
"what's the weather"             allowed=True  layer=classifier  


Two independent layers. A restricted topic slips through only if both miss it.

## The silence problem

Blocking works. But watch what the prospect experiences:


In [7]:
def handle_v1(message):
    v = check_scope(message)
    if not v['allowed']:
        return None            # queued for a human, nothing sent
    return fake_agent(message, [])

print(repr(handle_v1("what's the damage")))


None


`None`. They get nothing.

A pricing question is the highest-intent message in the entire funnel, and the system answers it with silence until I happen to check the queue. They go cold while waiting.

Fix: a fixed acknowledgement, sent immediately. Safe to automate because it contains no factual claim — it's a constant, not a generation, so it can't drift.


In [8]:
HOLDING = {
    'pricing': "Good question - I'd rather get you exact numbers than guess. "
               'Someone will text you shortly with the details.',
    'contract_terms': 'Fair thing to ask before committing to anything. '
                      "Let me get you a straight answer - someone will follow up shortly.",
    'legal_compliance': "That's worth answering precisely rather than off the cuff. "
                        "I'll have someone get back to you shortly.",
    'asked for a person': "Of course - I'll have someone from the team reach out shortly.",
}

def handle_v2(message):
    v = check_scope(message)
    if not v['allowed']:
        return HOLDING.get(v['reason'], '')
    return fake_agent(message, [])

for m in ["what's the damage", 'can I talk to someone', 'do I get stuck in something']:
    print(f'{m!r:32}\n  -> {handle_v2(m)}\n')


"what's the damage"             
  -> Good question - I'd rather get you exact numbers than guess. Someone will text you shortly with the details.

'can I talk to someone'         
  -> Of course - I'll have someone from the team reach out shortly.

'do I get stuck in something'   
  -> Fair thing to ask before committing to anything. Let me get you a straight answer - someone will follow up shortly.



One rule I put in a test: **no digits in a holding reply**. These bypass the grounding check precisely because they assert nothing, so the moment a number appears in one, that assumption is broken and an unverified claim goes out unchecked. A digit is the cheapest canary for that.


In [9]:
for k, v in HOLDING.items():
    assert not any(ch.isdigit() for ch in v), f'{k} contains a number'
print('all holding replies are claim-free')


all holding replies are claim-free


## The output side

Input guarding handles topics I know are dangerous. It does nothing about the interesting failure: a question that IS in scope, answered with a detail that isn't in the KB.


In [10]:
ENTRIES = [{'id': 'is_it_a_robot',
            'answer': "Yes, it's an AI voice agent, not a recording."}]

def fake_agent_2(message, entries):
    if not entries:
        return 'I can only help with VoiceCaptures questions.'
    # The model rephrases the entry... and adds something that isn't in it.
    return "Yes, it's AI - and it books appointments straight into your calendar."

draft = fake_agent_2('is this a robot', ENTRIES)
print(draft)


Yes, it's AI - and it books appointments straight into your calendar.


Calendar booking might even be true. But it isn't in the entry the agent was given, so nothing verified it. That's the definition of a claim I can't stand behind.

So: a second model reads the draft against the entries and asks whether every factual claim is supported.


In [11]:
def fake_grounding_check(draft, entries):
    """Real one is a cheap model. This fakes its judgement."""
    supported = ' '.join(e['answer'].lower() for e in entries)
    # Refusals, greetings and questions assert nothing - grounded by definition.
    low = draft.lower()
    if 'can only help' in low or low.endswith('?'):
        return {'grounded': True, 'unsupported': ''}
    if 'calendar' in low and 'calendar' not in supported:
        return {'grounded': False, 'unsupported': 'books appointments into your calendar'}
    return {'grounded': True, 'unsupported': ''}

print(fake_grounding_check(draft, ENTRIES))
print(fake_grounding_check('I can only help with VoiceCaptures questions.', []))


{'grounded': False, 'unsupported': 'books appointments into your calendar'}
{'grounded': True, 'unsupported': ''}


That second case matters and I got it wrong at first.

My original instruction to the judge said: if the entries are empty, any factual claim about the product is unsupported. So when the agent correctly refused an off-topic question with 'I can only help with VoiceCaptures questions', the judge flagged it — it mentions the product, so it looked like a claim.

It isn't. Naming something while declining to discuss it asserts nothing. Refusals, greetings and questions are grounded by definition.

## The ordering problem I didn't see coming

This is the part that changed the architecture.

I originally gave the agent a `send_sms` tool, so it could reply directly. Output guardrails run *after* tool calls. Which means:


In [12]:
def agent_with_send_tool(message, entries, send):
    draft = fake_agent_2(message, entries)
    send(draft)                # agent decides to send, mid-run
    return draft

sent = []
draft = agent_with_send_tool('is this a robot', ENTRIES, sent.append)
verdict = fake_grounding_check(draft, ENTRIES)
print('guardrail says grounded:', verdict['grounded'])
print('already delivered      :', sent)


guardrail says grounded: False
already delivered      : ["Yes, it's AI - and it books appointments straight into your calendar."]


The check correctly caught it. Twilio had already delivered it. You cannot unsend a text.

The guardrail wasn't wrong -- it was in a position where being right didn't help. So the agent lost the send tool. It returns text; code
decides what happens to it.


In [13]:
def handle_v3(message, entries):
    v = check_scope(message)
    if not v['allowed']:
        return ('blocked_scope', HOLDING.get(v['reason'], ''))

    draft = fake_agent_2(message, entries)          # no tools, just returns
    verdict = fake_grounding_check(draft, entries)
    if not verdict['grounded']:
        return ('blocked_grounding', f"queued for review: {verdict['unsupported']}")

    return ('sent', draft)                          # code sends, after checks

for m, e in [('is this a robot', ENTRIES), ("what's the damage", ENTRIES),
             ('what is the weather', [])]:
    action, detail = handle_v3(m, e)
    print(f'{m!r:24} {action:18} {detail[:60]}')


'is this a robot'        blocked_grounding  queued for review: books appointments into your calendar
"what's the damage"      blocked_scope      Good question - I'd rather get you exact numbers than guess.
'what is the weather'    sent               I can only help with VoiceCaptures questions.


Same reasoning killed the handoff tool later. It called `set_autopilot(False)` as a side effect, so the agent could silently
switch autopilot off on any message it chose, with nothing between the decision and the effect and no record it happened. It's now a boolean on the returned object — same signal, but code acts on it and the trace shows it.

General rule I'd take from this: **an agent should return decisions, not perform them.** Anything irreversible belongs in code, after the checks.

## What this became

| here | in the repo |
|---|---|
| `check_keywords` | `check_scope_keywords` in `app/agents/guardrails.py` |
| `fake_classifier` | `_scope_agent`, a cheap model with structured output |
| `check_scope` | `check_scope`, same two-layer shape |
| `fake_grounding_check` | `check_grounding` |
| `HOLDING` | `holding_reply` fields in `voicecaptures.yaml` |
| `handle_v3` | `run_autopilot` in `app/agents/autopilot.py` |

Differences worth knowing. The real classifier's instructions are *generated from the YAML*, so adding a restricted topic updates both
layers — hardcoding the list would let them drift. And the real one fails closed on any exception, which costs nothing since autopilot can't draft a reply without the model API either way.

Look at the real thing:


In [14]:
import sys, os
sys.path.insert(0, os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else '.')
for k, v in {'SUPABASE_URL':'https://t.supabase.co','SUPABASE_SERVICE_KEY':'t',
             'OPENAI_API_KEY':'sk-t','TWILIO_ACCOUNT_SID':'ACt',
             'TWILIO_AUTH_TOKEN':'t','TWILIO_FROM_NUMBER':'+15550000000'}.items():
    os.environ.setdefault(k, v)

from app.kb.loader import match_restricted, wants_human, holding_reply_for

for m in ['how much does it cost', 'is there a contract', 'can I talk to someone']:
    r = match_restricted(m)
    reply = holding_reply_for(r.id) if r else ('handoff' if wants_human(m) else '-')
    print(f'{m!r:28}\n  {reply}\n')


'how much does it cost'     
  Good question - I'd rather get you exact numbers than guess. Someone will text you shortly with the details.

'is there a contract'       
  Fair thing to ask before committing to anything. Let me get you a straight answer on that - someone will follow up shortly.

'can I talk to someone'     
  handoff



The classifier layer and grounding judge both need a real API key, so I haven't run them here. To try them:

```python
from app.agents.guardrails import check_scope, check_grounding
await check_scope("what's the damage")
```

Costs a fraction of a cent per call.

## Still open

Neither model-based check is measured. I have no false-positive rate for the classifier and no accuracy number for the judge — an LLM grading an LLM is just a second opinion until it's checked against labels.

'Err toward RESTRICTED' is a real instruction with a real cost, and I don't know how much it over-blocks. That's what the golden set is for.
